# Task 2: Sentence Ordering + Translation (final)

**Input:** punctuated Hindi text from Task 1 (`sample-N_punctuated.txt`). The sentences are still jumbled.
**Output:** the sentences in a sensible order, grouped in chunks of 5, each chunk translated to English.

**Rules of the task:** no off-the-shelf LLM (GPT / Gemini / Claude) and **every model must use less than 1.5 GB of GPU RAM**.

## How the 1.5 GB rule is met
* **Run on a GPU** (Runtime > Change runtime type > T4 GPU). The notebook stops with a message if there is none, because the rule cannot be checked on a CPU.
* **Hard limit:** `torch.cuda.set_per_process_memory_fraction` is set to 1.5 GB, so PyTorch **cannot** allocate more. If a model were too big it would fail instead of silently using more memory.
* **One model at a time:** each model is deleted before the next one is loaded.
* **Two measurements per stage:** PyTorch's own peak, and the **whole GPU including the CUDA context** (what `nvidia-smi` would show). Both are printed and saved to `gpu_memory_report.csv`.

| Stage | Model | How it stays small |
|---|---|---|
| 1. Sentence meaning | `paraphrase-multilingual-MiniLM-L12-v2` | fp16 (~0.24 GB) |
| 2. Which sentence follows which | **our own model**: MiniLM fine-tuned as a cross-encoder on Hindi Wikipedia | the 250k-word table stays in CPU RAM, only ~21 M weights are trained |
| 3. Translation | `facebook/nllb-200-distilled-600M` in 8-bit (fallbacks: 4-bit, then `m2m100_418M`) | quantised; the first option that really measures under 1.5 GB is used |

TF-IDF and the clustering run on the CPU and use no GPU memory.

## Pipeline
1. Clean the text (remove `NULL`, fix broken numbers, unify Hindi spelling) and split into sentences
2. MiniLM embeddings + TF-IDF similarity, topic clustering
3. Learn the *direction* of the text ("does B come right after A?") from real Hindi, and measure it on held-out text
4. Find the best order inside every topic group, then cut into chunks of 5
5. Translate sentence by sentence, join 5 per chunk
6. Save the results, a list of sentences to double-check and the GPU memory report

**Time:** about 10 to 15 minutes on a T4.

## 1. Setup

In [1]:
!pip install -q sentence-transformers sentencepiece bitsandbytes accelerate datasets

In [2]:
import os, re, glob, json, zipfile
import numpy as np

# ---- Google Drive (only works inside Colab) ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_DIR = "/content/drive/MyDrive/Research_Project" if IN_COLAB else "."

# If your Task 1 outputs are somewhere else, put that folder here:
MANUAL_DIR = None

# ---- find the folder that has the *_punctuated.txt files ----
candidates = [
    MANUAL_DIR,
    os.path.join(PROJECT_DIR, "Model_1", "punctuation_outputs"),
    os.path.join(PROJECT_DIR, "punctuation_outputs"),
]
PUNCTUATED_DIR = None
for folder in candidates:
    if folder and glob.glob(os.path.join(folder, "*_punctuated.txt")):
        PUNCTUATED_DIR = folder
        break

# ---- fallback: upload the zip of Task 1 outputs by hand ----
if PUNCTUATED_DIR is None and IN_COLAB:
    from google.colab import files
    print("Task 1 outputs not found on Drive. Please upload the zip file:")
    uploaded = files.upload()
    PUNCTUATED_DIR = "/content/punctuated_uploaded"
    os.makedirs(PUNCTUATED_DIR, exist_ok=True)
    for name in uploaded:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                z.extractall(PUNCTUATED_DIR)
        else:
            os.replace(name, os.path.join(PUNCTUATED_DIR, name))

assert PUNCTUATED_DIR is not None, "Could not find *_punctuated.txt files"

OUTPUT_DIR = os.path.join(PROJECT_DIR, "Model_2", "Sentence_Ordering")
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(PUNCTUATED_DIR, "**", "*_punctuated.txt"), recursive=True))
print("Reading from:", PUNCTUATED_DIR)
print("Saving to   :", OUTPUT_DIR)
print("Files found :", len(files))
for f in files:
    print("  ", os.path.basename(f))

Mounted at /content/drive
Reading from: /content/drive/MyDrive/Research_Project/Model_1/punctuation_outputs
Saving to   : /content/drive/MyDrive/Research_Project/Model_2/Sentence_Ordering
Files found : 7
   Copy of sample-1_punctuated.txt
   Copy of sample-2_punctuated.txt
   Copy of sample-3_punctuated.txt
   Copy of sample-4_punctuated.txt
   Copy of sample-5_punctuated.txt
   Copy of sample-6_punctuated.txt
   Copy of sample-7_punctuated.txt


### GPU check and memory tools

In [3]:
import torch, gc, time, random, unicodedata

GPU_LIMIT_GB = 1.5          # limit given in the task
REQUIRE_GPU  = True         # the rule can only be checked on a GPU (set False only for a quick CPU test)
GB = 1024 ** 3

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu" and REQUIRE_GPU:
    raise RuntimeError("No GPU found. Use Runtime > Change runtime type > T4 GPU and run everything again. "
                       "(Only for a quick test you can set REQUIRE_GPU = False above.)")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

gpu_log = []     # one row per stage, saved at the end as gpu_memory_report.csv

if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / GB
    # HARD LIMIT: PyTorch raises an out-of-memory error instead of using more than 1.5 GB
    torch.cuda.set_per_process_memory_fraction(min(1.0, GPU_LIMIT_GB / total_gb), 0)
    print(f"GPU: {props.name} ({total_gb:.1f} GB in total)")
    print(f"Hard limit set: this notebook can never allocate more than {GPU_LIMIT_GB} GB through PyTorch.")
else:
    print("WARNING: running on CPU, GPU memory is not measured.")

def free_gpu():
    # delete what is no longer needed and reset the peak counter
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def device_used_gb():
    # everything that sits on the GPU right now, INCLUDING the CUDA context that PyTorch does not count
    free, total = torch.cuda.mem_get_info()
    return (total - free) / GB

def report_gpu(stage):
    if device != "cuda":
        print(f"[{stage}] CPU run - no GPU memory measured")
        return None
    torch_peak = torch.cuda.max_memory_allocated() / GB
    whole = device_used_gb()
    row = dict(stage=stage, pytorch_peak_gb=round(torch_peak, 3), whole_gpu_gb=round(whole, 3),
               pytorch_under_limit=bool(torch_peak < GPU_LIMIT_GB), whole_gpu_under_limit=bool(whole < GPU_LIMIT_GB))
    gpu_log.append(row)
    verdict = "OK" if whole < GPU_LIMIT_GB else "above 1.5 GB when the CUDA context is counted"
    print(f"[{stage}]")
    print(f"    PyTorch peak {torch_peak:.2f} GB | whole GPU incl. CUDA context {whole:.2f} GB | limit {GPU_LIMIT_GB} GB -> {verdict}")
    return row

free_gpu()
idle = device_used_gb() if device == "cuda" else 0.0
print(f"GPU in use before any model is loaded (CUDA context only): {idle:.2f} GB")

GPU: Tesla T4 (14.6 GB in total)
Hard limit set: this notebook can never allocate more than 1.5 GB through PyTorch.
GPU in use before any model is loaded (CUDA context only): 0.10 GB


### Settings
Everything you might want to tune is in this one cell.

In [4]:
CHUNK_SIZE        = 5      # task says: combine 5 sentences
MAX_CLUSTERS      = 3      # most topics we try to find in one sample
MIN_CLUSTER_SIZE  = 6      # a topic group smaller than this is not trusted
MIN_SILHOUETTE    = 0.08   # if clusters are not clearly separated, treat sample as ONE topic
SCORE_TOLERANCE   = 0.02   # prefer FEWER clusters when scores are almost equal
USE_CLUSTERING    = True   # set False to order each sample as one single topic
N_START_TRIES     = 8      # how many different first-sentences the ordering search tries
LONG_SENTENCE     = 40     # sentences longer than this (in words) are flagged

# --- learned ordering model (section 6b) ---
USE_LEARNED       = True   # learn "does sentence B follow sentence A?" from real Hindi text
WIKI_ARTICLES     = 800    # Hindi Wikipedia articles used for learning
WIKI_SENTS        = 20     # sentences taken from the start of every article
PAIR_EPOCHS       = 2      # training passes over the sentence pairs
PAIR_BATCH        = 16     # pairs per training step (small = low GPU memory)
PAIR_MAXLEN       = 128    # tokens of "sentence A + sentence B"
MAX_TRAIN_PAIRS   = 32000  # lower this if you are short on time
WINDOW            = 10     # size of the shuffled test windows used for evaluation
N_EVAL_WINDOWS    = 80     # test windows (half for choosing the method, half for the final number)

# --- translation ---
USE_GLOSSARY      = True   # fix names/terms the translation model gets wrong (see section 8)
TRANSLATE_MODE    = "sentence"   # "sentence" = translate each sentence, then join the 5 (best quality)
                                 # "chunk"    = translate the 5 joined sentences as one text

## 2. Load text, remove `NULL`, split into sentences

Task 1 left some `NULL` tokens and some punctuation inside numbers (`350- -400`, `7, ,000`), so we clean those first.
We also **unify the Unicode spelling** of letters like ढ़ (the files contain two different encodings of the same letter, which broke word matching).

In [5]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)                  # one spelling for letters like ढ़
    text = re.sub(r"\bNULL\b\s*[-:]?", " ", text)              # remove NULL (and a dash right after it)
    # Task 1 sometimes put punctuation inside numbers: "350- -400", "7, ,000", "4. 5"
    text = re.sub(r"(\d)-\s+-(\d)", r"\1-\2", text)            # 350- -400  -> 350-400
    text = re.sub(r"(\d),\s+,(\d)", r"\1,\2", text)            # 7, ,000    -> 7,000
    text = re.sub(r"(\d)\.\s+(\d)", r"\1.\2", text)            # 4. 5       -> 4.5
    text = re.sub(r"\(\s*\)", " ", text)                       # empty brackets "( )"
    text = re.sub(r"\(\s+", "(", text)                         # "( Bombyx" -> "(Bombyx"
    text = re.sub(r"\s+", " ", text)                           # fix extra spaces
    return text.strip()

def split_sentences(text):
    # split after danda / ? / !  and after a full stop that is followed by Hindi text
    parts = re.split(r"(?<=[।?!])\s+|(?<=\.)\s+(?=[ऀ-ॿ])", text)
    parts = [p.strip() for p in parts if p.strip()]

    # glue very short fragments to the next sentence
    sentences, buffer = [], ""
    for p in parts:
        p = (buffer + " " + p).strip() if buffer else p
        if len(p.split()) < 3:
            buffer = p
        else:
            sentences.append(p)
            buffer = ""
    if buffer:                       # leftover fragment at the very end
        if sentences:
            sentences[-1] += " " + buffer
        else:
            sentences.append(buffer)
    return sentences

sentence_data = {}   # file name -> list of sentences
for path in files:
    name = os.path.basename(path)
    with open(path, encoding="utf-8") as f:
        raw = f.read()
    sentence_data[name] = split_sentences(clean_text(raw))

    long_ones = [s for s in sentence_data[name] if len(s.split()) > LONG_SENTENCE]
    print(f"{name:28} {len(sentence_data[name]):3d} sentences | {len(long_ones)} longer than {LONG_SENTENCE} words")

Copy of sample-1_punctuated.txt  49 sentences | 0 longer than 40 words
Copy of sample-2_punctuated.txt  49 sentences | 0 longer than 40 words
Copy of sample-3_punctuated.txt  48 sentences | 1 longer than 40 words
Copy of sample-4_punctuated.txt  50 sentences | 1 longer than 40 words
Copy of sample-5_punctuated.txt  51 sentences | 0 longer than 40 words
Copy of sample-6_punctuated.txt  35 sentences | 1 longer than 40 words
Copy of sample-7_punctuated.txt  48 sentences | 4 longer than 40 words


In [6]:
# quick look at the first 3 sentences of every file
for name, sents in sentence_data.items():
    print("\n", name)
    for i, s in enumerate(sents[:3], 1):
        print(f"  {i}. {s}")


 Copy of sample-1_punctuated.txt
  1. भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।
  2. चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है।
  3. द्वारका चार धामों में एक धाम भी है।

 Copy of sample-2_punctuated.txt
  1. प्रमुख स्थानों पर क्लोज सर्किट टीवी कैमरे भी लगाए गए हैं।
  2. गंगा तट के रेतीले मैदान में मेला लगता है, किंतु गढ़ मुक्तेश्वर के गंगा मंदिर में अब 86 सीढ़ियाँ ही बची हैं।
  3. मेला स्थल दिल्ली से लगभग 90 किलोमीटर पड़ता है।

 Copy of sample-3_punctuated.txt
  1. कश्मीरी गेट से लगातार मेरठ तक की बसें जाती हैं।
  2. श्रेयंषकुमार की स्मृति में रत्न स्तम्भ बनवाया था।
  3. बताते हैं कि तिजारा क्षेत्र के एक मशहूर वैद्य बिहारी लाल की पत्‍नी सरस्वती देवी ने तीन दिन का उपवास रखा था।

 Copy of sample-4_punctuated.txt
  1. वैसे अन्य जाति के रेशम कीट 350-400 अंडे देते हैं।
  2. रेशम कई प्रकार के कीड़ों द्वारा बनाया जाता है परंतु इन सब में बांबिक्स मोरी(Bombyx Mori) जाति का रेशम कीड़ा ही प्रमुख है और सेरीकल्चर फार्म के विशेष वाता

## 3. Sentence similarity: MiniLM (meaning) + TF-IDF (words)

* **MiniLM embeddings** - sentences with similar meaning get similar vectors. Loaded in **fp16** on the GPU (stage 1).
* **TF-IDF** - sentences sharing the same rare words (names of places, people) score high. A Hindi-aware tokenizer is used because the default one cuts Hindi words at the vowel signs.

In [7]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

free_gpu()
embedder = SentenceTransformer(MODEL_NAME, device=device)
if device == "cuda":
    embedder.half()          # fp16 = half the memory (about 0.24 GB instead of 0.47 GB)
print("MiniLM loaded on", device, "(fp16)" if device == "cuda" else "")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MiniLM loaded on cuda (fp16)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

hindi_stopwords = [
    "के", "की", "का", "से", "में", "पर", "को", "है", "हैं", "था", "थे", "थी", "भी",
    "और", "यह", "वह", "इस", "उस", "तो", "ही", "एक", "ने", "लिए", "तक", "जो", "कि",
    "या", "तथा", "अपने", "अपनी", "अपना", "वे", "ये", "कर", "करने", "होता", "होती",
    "होते", "हो", "गया", "गई", "गए", "जाता", "जाती", "जाते", "रहा", "रही", "रहे",
    "सकते", "सकता", "द्वारा", "जब", "तब", "बाद", "साथ", "बहुत", "कुछ", "कोई",
]

# a Hindi word = Devanagari letters + vowel signs, but NOT the danda (। ॥)
HINDI_TOKEN = r"[\u0900-\u0963\u0966-\u097F]+|[A-Za-z]+"

def tfidf_similarity(sentences):
    # word-level TF-IDF (1 and 2 word groups)
    word_vec = TfidfVectorizer(token_pattern=HINDI_TOKEN, stop_words=hindi_stopwords,
                               ngram_range=(1, 2), sublinear_tf=True)
    word_sim = cosine_similarity(word_vec.fit_transform(sentences))

    # character-level TF-IDF (2 to 4 letters)
    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
    char_sim = cosine_similarity(char_vec.fit_transform(sentences))

    return 0.5 * word_sim + 0.5 * char_sim

vectors  = {}   # file -> MiniLM embeddings (needed by the learned ordering model)
semantic = {}   # file -> MiniLM similarity matrix
lexical  = {}   # file -> TF-IDF similarity matrix

for name, sents in sentence_data.items():
    emb = embedder.encode(sents, batch_size=16, convert_to_numpy=True,
                          normalize_embeddings=True, show_progress_bar=False)
    vectors[name]  = emb.astype(np.float32)
    semantic[name] = cosine_similarity(emb.astype(np.float32))
    lexical[name]  = tfidf_similarity(sents)
    print(f"{name:28} similarity matrices: {semantic[name].shape}")

Copy of sample-1_punctuated.txt similarity matrices: (49, 49)
Copy of sample-2_punctuated.txt similarity matrices: (49, 49)
Copy of sample-3_punctuated.txt similarity matrices: (48, 48)
Copy of sample-4_punctuated.txt similarity matrices: (50, 50)
Copy of sample-5_punctuated.txt similarity matrices: (51, 51)
Copy of sample-6_punctuated.txt similarity matrices: (35, 35)
Copy of sample-7_punctuated.txt similarity matrices: (48, 48)


In [9]:
# sanity check: for 3 sentences of sample-1, show the closest sentences
name = list(sentence_data)[0]
sents, sem, lex = sentence_data[name], semantic[name], lexical[name]
combined = 0.6 * np.clip(sem, 0, 1) + 0.4 * lex

for i in range(3):
    scores = combined[i].copy()
    scores[i] = -1
    best = np.argsort(scores)[::-1][:2]
    print(f"\nSentence {i+1}: {sents[i]}")
    for j in best:
        print(f"   close ({scores[j]:.2f}) #{j+1}: {sents[j]}")


Sentence 1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।
   close (0.52) #22: महाराज शिव ने भगवान परशुराम से यहाँ शिव मंदिर की स्थापना कराई थी।
   close (0.52) #33: गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।

Sentence 2: चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है।
   close (0.76) #16: चेंगलपट स्टेशन से एक लाइन अरकोनम तक जाती है।
   close (0.57) #45: उत्तर रेलवे की मुगलसराय से अमृतसर जाने वाली मुख्य लाइन पर स्टेशन है।

Sentence 3: द्वारका चार धामों में एक धाम भी है।
   close (0.44) #42: इसलिए हरिद्वार को गंगा द्वार भी कहते हैं।
   close (0.40) #26: चलें गढ़ मुक्तेश्वर का गंगा मेला लक्सर से एक लाइन हरिद्वार तक गई है।


## 4. Topic clustering

Some samples look like **two or more documents mixed together** (for example railway routes + a fair, or cotton + silk). If we order everything as one text, the topics get tangled.
So first we group sentences by topic with *hierarchical (Ward) clustering* and pick the number of groups with the **silhouette score**. A group smaller than `MIN_CLUSTER_SIZE` is not trusted, and if no split is clearly good the sample is treated as one topic.

In [10]:
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score

def find_topic_clusters(sem, lex):
    n = len(sem)
    sim = 0.6 * np.clip(sem, 0, 1) + 0.4 * lex       # how related is each pair of sentences
    dist = 1 - sim
    dist = (dist + dist.T) / 2                       # make symmetric
    np.fill_diagonal(dist, 0)

    if not USE_CLUSTERING:
        return np.zeros(n, dtype=int), {}

    # Ward linkage on the similarity rows: each sentence is described by how similar
    # it is to every other sentence. This gives balanced groups.
    tree = linkage(sim, method="ward")

    scores = {}
    for k in range(2, MAX_CLUSTERS + 1):
        labels = fcluster(tree, k, criterion="maxclust")
        sizes = np.bincount(labels)[1:]
        if len(sizes) < 2 or sizes.min() < MIN_CLUSTER_SIZE:
            continue                      # some group is too small, skip this k
        scores[k] = silhouette_score(dist, labels, metric="precomputed")

    best_k = 1
    if scores:
        top = max(scores.values())
        if top >= MIN_SILHOUETTE:
            # among the almost-best scores take the SMALLEST number of clusters
            best_k = min(k for k, v in scores.items() if v >= top - SCORE_TOLERANCE)

    if best_k == 1:
        labels = np.zeros(n, dtype=int)
    else:
        labels = fcluster(tree, best_k, criterion="maxclust") - 1
    return labels, scores

clusters = {}
for name in sentence_data:
    labels, scores = find_topic_clusters(semantic[name], lexical[name])
    clusters[name] = labels
    sizes = np.bincount(labels).tolist()
    score_text = ", ".join(f"k={k}: {v:.2f}" for k, v in scores.items()) or "no valid split"
    print(f"{name:28} -> {len(sizes)} topic(s), sizes {sizes} | silhouette {score_text}")

Copy of sample-1_punctuated.txt -> 2 topic(s), sizes [13, 36] | silhouette k=2: 0.22, k=3: 0.21
Copy of sample-2_punctuated.txt -> 1 topic(s), sizes [49] | silhouette k=2: 0.07, k=3: 0.04
Copy of sample-3_punctuated.txt -> 1 topic(s), sizes [48] | silhouette k=2: 0.08, k=3: 0.07
Copy of sample-4_punctuated.txt -> 2 topic(s), sizes [19, 31] | silhouette k=2: 0.11, k=3: 0.11
Copy of sample-5_punctuated.txt -> 2 topic(s), sizes [30, 21] | silhouette k=2: 0.10
Copy of sample-6_punctuated.txt -> 1 topic(s), sizes [35] | silhouette k=2: 0.05, k=3: 0.07
Copy of sample-7_punctuated.txt -> 1 topic(s), sizes [48] | silhouette k=2: 0.04


In [11]:
# look inside the clusters to check they make sense (first 3 sentences of each)
for name, labels in clusters.items():
    if labels.max() == 0:
        continue
    print("\n" + "=" * 70, "\n", name)
    for c in range(labels.max() + 1):
        members = np.where(labels == c)[0]
        print(f"\n  Topic {c} ({len(members)} sentences)")
        for i in members[:3]:
            print("    -", sentence_data[name][i][:90])


 Copy of sample-1_punctuated.txt

  Topic 0 (13 sentences)
    - चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है।
    - दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जाने वाली मुख्य लाइन पर चेन्नै से 35 मील की दूर
    - पूर्वोत्तर रेलवे की एक लाइन भटनी से तथा दूसरी छपरा से इलाहाबाद सिटी तक जाती है।

  Topic 1 (36 sentences)
    - भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।
    - द्वारका चार धामों में एक धाम भी है।
    - कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती।

 Copy of sample-4_punctuated.txt

  Topic 0 (19 sentences)
    - यह बीज कपास की फसल को बीमारियों और कीड़ों से बचाने की शक्‍ति प्रदान करता है।
    - कपास के रेशों के टेंट को दिसंबर माह में भारत में प्रायः हाथ से तथा विदेशों में मशीन से तोड
    - अभी हाल ही में कुछ वर्ष पूर्व आनुवंशिक परिवर्तन(genetic modification) द्वारा अमेरिका में क

  Topic 1 (31 sentences)
    - वैसे अन्य जाति के रेशम कीट 350-400 अंडे देते हैं।
    - रेशम कई प्रक

## 5. Which sentence comes after which? (hand-written Hindi clues)

Linking words such as `फिर`, `इसलिए`, `उक्त` show that a sentence needs a sentence before it. Opening phrases (`सबसे पहले`) make good first sentences and closing phrases (`इस प्रकार`) good last ones.
These rules are weak on their own, so section 6b adds a learned model.

In [12]:
# words that show a sentence depends on an earlier one (strong = clearer clue)
STRONG_LINKERS = ["इसके बाद", "इसके पश्चात", "इसके अलावा", "इसके अतिरिक्त", "साथ ही", "बाद में",
                  "उस समय", "फिर", "तब", "इसलिए", "अतः", "इसी कारण", "इस कारण", "इसी", "उक्त",
                  "लेकिन", "किंतु", "परंतु", "मगर", "वहाँ", "यहाँ", "इस प्रकार", "अंत में"]
WEAK_LINKERS   = ["यह", "वह", "इस", "उस", "ये", "वे", "इन", "उन", "उसके", "उनके", "इसके",
                  "इसका", "इसकी", "उसका", "उसकी", "और", "तथा", "जब", "जिसमें", "जिसे"]

OPENING_PHRASES = ["सबसे पहले", "शुरुआत में", "आरंभ में", "प्रारंभ में"]
CLOSING_PHRASES = ["इस प्रकार", "अंत में", "अंततः", "अंतिम रूप से"]

def starts_with_phrase(sentence, phrases):
    s = sentence.strip() + " "
    return any(s.startswith(p + " ") for p in phrases)

def get_clues(sentences):
    n = len(sentences)
    need_context = np.zeros(n)   # 0 = standalone, 0.2 = weak linker, 0.5 = strong linker
    open_bonus   = np.zeros(n)
    close_bonus  = np.zeros(n)
    for i, s in enumerate(sentences):
        if starts_with_phrase(s, STRONG_LINKERS):
            need_context[i] = 0.5
        elif starts_with_phrase(s, WEAK_LINKERS):
            need_context[i] = 0.2
        if starts_with_phrase(s, OPENING_PHRASES):
            open_bonus[i] = 0.4
        if starts_with_phrase(s, CLOSING_PHRASES):
            close_bonus[i] = 0.3
    return need_context, open_bonus, close_bonus

def build_scores(sem, lex, sentences):
    need_context, open_bonus, close_bonus = get_clues(sentences)

    base = 0.6 * np.clip(sem, 0, 1) + 0.4 * lex            # how related are i and j
    T = base * (1 + need_context[None, :])                  # linker sentences must match their neighbour well
    np.fill_diagonal(T, -1)

    centrality = base.mean(axis=1)
    centrality = (centrality - centrality.mean()) / (centrality.std() + 1e-9)

    start_score = open_bonus - need_context + 0.1 * centrality   # good first sentence?
    end_score   = close_bonus                                     # good last sentence?
    return T, start_score, end_score

# test on one file: which sentences does the model like as openers?
name = list(sentence_data)[0]
T, start_s, end_s = build_scores(semantic[name], lexical[name], sentence_data[name])
print(name)
for i in np.argsort(start_s)[::-1][:3]:
    print(f"  opener candidate #{i+1} (score {start_s[i]:.2f}): {sentence_data[name][i][:80]}")

Copy of sample-1_punctuated.txt
  opener candidate #26 (score 0.19): चलें गढ़ मुक्तेश्वर का गंगा मेला लक्सर से एक लाइन हरिद्वार तक गई है।
  opener candidate #21 (score 0.16): ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है।
  opener candidate #33 (score 0.15): गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।


## 6. Finding the best order (path search)

We look for the order with the highest total score: start score + sum of "B after A" scores + end score.
Trying all orders is impossible (50! options), so: (1) build greedily from several start sentences, (2) improve by moving single sentences to better positions until nothing improves.

In [13]:
def path_score(path, T, start_s, end_s):
    path = np.array(path)
    total = start_s[path[0]] + end_s[path[-1]]
    if len(path) > 1:
        total += T[path[:-1], path[1:]].sum()
    return total

def greedy_path(first, T):
    n = len(T)
    path = [first]
    left = set(range(n)) - {first}
    while left:
        last = path[-1]
        nxt = max(left, key=lambda j: T[last, j])
        path.append(nxt)
        left.remove(nxt)
    return path

def improve_path(path, T, start_s, end_s, max_sweeps=15):
    path = list(path)
    best = path_score(path, T, start_s, end_s)
    for _ in range(max_sweeps):
        changed = False
        for item in list(path):
            rest = [x for x in path if x != item]
            for pos in range(len(rest) + 1):
                trial = rest[:pos] + [item] + rest[pos:]
                score = path_score(trial, T, start_s, end_s)
                if score > best + 1e-9:
                    best, path, changed = score, trial, True
                    break          # take the first improvement, then continue
        if not changed:
            break
    return path, best

def order_group(T, start_s, end_s):
    # returns the best order (list of local indices) for one group of sentences
    n = len(T)
    if n == 1:
        return [0], 0.0
    starts = np.argsort(start_s)[::-1][:N_START_TRIES]
    best_path, best_score = None, -1e9
    for s in starts:
        path, score = improve_path(greedy_path(int(s), T), T, start_s, end_s)
        if score > best_score:
            best_path, best_score = path, score
    return best_path, best_score

## 6b. Learn the direction from real Hindi text (our own model)

MiniLM embeddings say two sentences are *related*, but not which one comes first (in the first version the direction accuracy was 52%, a coin flip).
So here we train our own **cross-encoder**: MiniLM reads *"sentence A + sentence B" together* and outputs one number, "B comes right after A". Reading both sentences together lets it use cues such as pronouns, linking words and repeated names.

* **Training data:** correctly ordered Hindi Wikipedia articles (streamed). True neighbours are label 1; the same two sentences in reverse order, sentences from the same article that are not neighbours, and sentences of other articles are label 0. The *reversed* pairs are what teaches direction.
* **GPU memory:** the 250k-word table (the biggest part of MiniLM) is frozen and stays in CPU RAM. Only the 12 transformer layers (~21 M weights) are trained, with a small batch and half precision.
* **Start model:** a small logistic regression that scores how much a sentence looks like the first sentence of a text (CPU only).
* **Honest evaluation:** on articles the model never saw we shuffle windows of 10 sentences, reorder them and measure **Kendall's tau** (1 = perfect, 0 = random). Half of the windows choose the best method, the other half give the number to report.

If anything fails (no Wikipedia, out of memory ...), the notebook falls back to the rule-based ordering and says so.

In [14]:
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import kendalltau
import pandas as pd
warnings.filterwarnings("ignore")

BEST = dict(use_learned=False)     # setting used for the real samples (changed below if learning helps)
start_model = None
L_samples = {}                     # file -> pair log-odds matrix from the cross-encoder (filled in below)

def start_logits(E):
    z = start_model.decision_function(E)
    return (z - z.mean()) / (z.std() + 1e-9)

def final_scores(L, E, sents, sem, lex, cfg):
    # rule-based scores (section 5), optionally mixed with the learned scores
    Th, sh, eh = build_scores(sem, lex, sents)
    if (not cfg["use_learned"]) or L is None:
        return Th, sh, eh
    T = L / 4 + cfg["heur_w"] * Th
    np.fill_diagonal(T, -1e3)
    start_s = cfg["heur_w"] * sh
    if cfg["start_w"] > 0 and start_model is not None:
        start_s = start_s + cfg["start_w"] * start_logits(E)
    end_s = cfg["heur_w"] * eh
    return T, start_s, end_s

In [15]:
# ---- load Hindi Wikipedia (streamed, only what we need) ----
def clean_wiki_sentences(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\([^)]*\)", " ", text)          # remove (...) parts
    text = re.sub(r"\[[^\]]*\]", " ", text)         # remove [..] parts
    out = []
    for line in text.split("\n"):
        if len(line.split()) < 8:                    # headings and empty lines
            continue
        for s in split_sentences(re.sub(r"\s+", " ", line)):
            if 6 <= len(s.split()) <= 50:
                out.append(s)
    return out

def load_hindi_articles(n_articles, n_sents, max_scan=10000):
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", "20231101.hi", split="train", streaming=True)
    ds = ds.shuffle(seed=42, buffer_size=1000)
    articles = []
    for k, row in enumerate(ds):
        if k >= max_scan or len(articles) >= n_articles:
            break
        sents = clean_wiki_sentences(row["text"])
        if len(sents) >= n_sents:
            articles.append(sents[:n_sents])
    return articles

articles = []
if USE_LEARNED:
    try:
        t0 = time.time()
        articles = load_hindi_articles(WIKI_ARTICLES, WIKI_SENTS)
        print(f"Loaded {len(articles)} articles ({sum(map(len, articles))} sentences) in {time.time() - t0:.0f}s")
        print("Example:", articles[0][0][:100])
    except Exception as e:
        print("Could not load Hindi Wikipedia:", type(e).__name__, str(e)[:150])
        print("-> continuing with the rule-based ordering only")

LEARNED_READY = len(articles) >= 100
print("Learned ordering model will be trained:", LEARNED_READY)

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Loaded 800 articles (16000 sentences) in 19s
Example: कोला प्रायद्वीप रूस के सुदूर पश्चिमोत्तरी भाग में स्थित एक प्रायद्वीप है।
Learned ordering model will be trained: True


In [16]:
# ---- embed the Wikipedia sentences, make test windows and training pairs, fit the start model ----
windows, tune_w, report_w = [], [], []
train_pairs, test_pairs, test_articles = [], [], []

def make_text_pairs(arts, seed, limit=None):
    rng = np.random.default_rng(seed)
    pairs = []
    for a, S in enumerate(arts):
        n = len(S)
        for i in range(n - 1):
            pairs.append((S[i], S[i + 1], 1))                  # the true next sentence
            pairs.append((S[i + 1], S[i], 0))                  # same two sentences, wrong direction
            j = int(rng.integers(0, n))
            while abs(j - i) < 2:
                j = int(rng.integers(0, n))
            pairs.append((S[i], S[j], 0))                      # same article, not neighbours
            b = int(rng.integers(0, len(arts)))
            while b == a:
                b = int(rng.integers(0, len(arts)))
            pairs.append((S[i], arts[b][int(rng.integers(0, len(arts[b])))], 0))   # a sentence of another article
    order = rng.permutation(len(pairs))
    pairs = [pairs[k] for k in order]
    return pairs[:limit] if limit else pairs

if LEARNED_READY:
    random.Random(0).shuffle(articles)
    n_test = max(40, len(articles) // 5)
    test_articles, train_articles = articles[:n_test], articles[n_test:]

    def embed_articles(arts):
        flat = [s for a in arts for s in a]
        e = embedder.encode(flat, batch_size=64, convert_to_numpy=True,
                            normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
        out, k = [], 0
        for a in arts:
            out.append(e[k:k + len(a)])
            k += len(a)
        return out

    t0 = time.time()
    train_emb = embed_articles(train_articles)
    test_emb = embed_articles(test_articles)
    print(f"Embedded {sum(map(len, train_articles)) + sum(map(len, test_articles))} sentences in {time.time() - t0:.0f}s")

    # start model: is this sentence the opening sentence of its article?
    Xs = np.vstack(train_emb)
    ys = np.concatenate([[1] + [0] * (len(E) - 1) for E in train_emb])
    start_model = LogisticRegression(C=0.5, max_iter=300, class_weight="balanced").fit(Xs, ys)
    ys_te = np.concatenate([[1] + [0] * (len(E) - 1) for E in test_emb])
    print(f"Start model test AUC: {roc_auc_score(ys_te, start_model.decision_function(np.vstack(test_emb))):.3f}")

    # test windows: 10 consecutive sentences of an unseen article, shuffled (we know the true order)
    rng = np.random.default_rng(123)
    for E, S in zip(test_emb, test_articles):
        for start in (0, WINDOW):
            if start + WINDOW <= len(S):
                perm = rng.permutation(WINDOW)               # shuffled item k really is sentence perm[k]
                idx = start + perm
                Ew, Sw = E[idx], [S[k] for k in idx]
                windows.append(dict(E=Ew, S=Sw, perm=perm,
                                    sem=np.clip(cosine_similarity(Ew), 0, 1), lex=tfidf_similarity(Sw)))
    rng.shuffle(windows)
    windows = windows[:N_EVAL_WINDOWS]
    half = len(windows) // 2
    tune_w, report_w = windows[:half], windows[half:]
    print(f"{len(tune_w)} windows for choosing the method, {len(report_w)} for the final report")

    train_pairs = make_text_pairs(train_articles, 0, MAX_TRAIN_PAIRS)
    test_pairs = make_text_pairs(test_articles, 1, 6000)
    print(f"training pairs: {len(train_pairs)} | test pairs: {len(test_pairs)}")

# ---- stage 1 is finished: measure the GPU and free MiniLM before the next model ----
report_gpu("Stage 1: MiniLM bi-encoder (fp16)")
embedder = None
free_gpu()

Embedded 16000 sentences in 13s
Start model test AUC: 0.900
40 windows for choosing the method, 40 for the final report
training pairs: 32000 | test pairs: 6000
[Stage 1: MiniLM bi-encoder (fp16)]
    PyTorch peak 0.62 GB | whole GPU incl. CUDA context 0.76 GB | limit 1.5 GB -> OK


In [17]:
# Stage 2 - small cross-encoder that learns "does sentence B come right after sentence A?"
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

class CPUEmbedding(nn.Module):
    # The word-embedding table of MiniLM is huge (250k words, ~0.37 GB) but we never train it.
    # We keep it in normal RAM and only move the few vectors of each batch to the GPU.
    # This keeps the GPU memory of this model very small.
    def __init__(self, embedding, gpu_device):
        super().__init__()
        self.weight = embedding.weight.data          # plain tensor (not a Parameter) -> stays on the CPU
        self.padding_idx = embedding.padding_idx
        self.gpu_device = gpu_device
    def forward(self, input_ids):
        vectors = F.embedding(input_ids.cpu(), self.weight, self.padding_idx)
        return vectors.to(self.gpu_device)

class PairScorer(nn.Module):
    # MiniLM reads "[sentence A] [sentence B]" together, then one number comes out:
    # high = "B follows A", low = "B does not follow A".
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.config.hidden_size, 1)
    def forward(self, **batch):
        hidden = self.encoder(**batch).last_hidden_state              # (batch, tokens, 384)
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)   # average over real tokens
        return self.head(pooled).squeeze(-1)

def build_pair_scorer():
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    encoder = AutoModel.from_pretrained(MODEL_NAME)
    encoder.embeddings.word_embeddings = CPUEmbedding(encoder.embeddings.word_embeddings, device)
    for p in encoder.embeddings.parameters():        # position / layer-norm tables: tiny, frozen too
        p.requires_grad = False
    return tok, PairScorer(encoder).to(device)

def tokenize_pairs(tok, a_list, b_list):
    enc = tok(a_list, b_list, truncation=True, max_length=PAIR_MAXLEN, padding=True, return_tensors="pt")
    return {k: v.to(device) for k, v in enc.items()}

def train_pair_scorer(tok, model, pairs, epochs, batch_size, lr=5e-5):
    encoder_params = [p for p in model.encoder.parameters() if p.requires_grad]
    head_params = list(model.head.parameters())
    optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": lr},
                                   {"params": head_params, "lr": lr * 10}], weight_decay=0.01)
    steps_per_epoch = (len(pairs) + batch_size - 1) // batch_size
    total_steps = epochs * steps_per_epoch
    warmup = max(1, int(0.06 * total_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: min((s + 1) / warmup, max(0.0, (total_steps - s) / max(1, total_steps - warmup))))
    use_amp = device == "cuda"                       # half precision on the GPU = faster and smaller
    scaler = torch.amp.GradScaler(device, enabled=use_amp)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(3.0, device=device))   # 1 true pair : 3 wrong pairs

    model.train()
    t0 = time.time()
    for epoch in range(epochs):
        order = np.random.default_rng(epoch).permutation(len(pairs))
        running, count = 0.0, 0
        for b in range(steps_per_epoch):
            batch = [pairs[k] for k in order[b * batch_size:(b + 1) * batch_size]]
            enc = tokenize_pairs(tok, [x[0] for x in batch], [x[1] for x in batch])
            y = torch.tensor([x[2] for x in batch], dtype=torch.float32, device=device)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits = model(**enc)
            loss = loss_fn(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(encoder_params + head_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item(); count += 1
            if (b + 1) % 500 == 0:
                print(f"  epoch {epoch + 1} step {b + 1}/{steps_per_epoch} loss {running / count:.3f} ({time.time() - t0:.0f}s)")
        print(f"epoch {epoch + 1}/{epochs} done | average loss {running / count:.3f} | {time.time() - t0:.0f}s")
    model.eval()

def score_pairs(tok, model, a_list, b_list, batch_size=128):
    out = []
    with torch.inference_mode():
        for s in range(0, len(a_list), batch_size):
            enc = tokenize_pairs(tok, a_list[s:s + batch_size], b_list[s:s + batch_size])
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
                logits = model(**enc)
            out.append(logits.float().cpu().numpy())
    return np.concatenate(out)

def pair_logit_matrix(tok, model, sents):
    # L[i, j] = how strongly the model believes that sentence j comes right after sentence i
    n = len(sents)
    ii, jj = np.where(~np.eye(n, dtype=bool))
    z = score_pairs(tok, model, [sents[i] for i in ii], [sents[j] for j in jj])
    L = np.full((n, n), -10.0, dtype=np.float32)
    L[ii, jj] = z
    return L

LEARNED_OK = False
pair_auc, direction_acc = None, None

if LEARNED_READY:
    tok = scorer = None
    try:
        free_gpu()
        tok, scorer = build_pair_scorer()
        n_train = sum(p.numel() for p in scorer.parameters() if p.requires_grad)
        print(f"trainable parameters: {n_train / 1e6:.1f} M (word table frozen and kept on the CPU)")
        print(f"training on {len(train_pairs)} sentence pairs for {PAIR_EPOCHS} epochs (batch {PAIR_BATCH}) ...")
        train_pair_scorer(tok, scorer, train_pairs, PAIR_EPOCHS, PAIR_BATCH)

        # --- test on articles the model never saw ---
        test_a = [p[0] for p in test_pairs]; test_b = [p[1] for p in test_pairs]
        test_y = np.array([p[2] for p in test_pairs])
        pair_auc = roc_auc_score(test_y, score_pairs(tok, scorer, test_a, test_b))
        fa, fb = [], []
        for S in test_articles:
            for i in range(len(S) - 1):
                fa.append(S[i]); fb.append(S[i + 1])
        direction_acc = float((score_pairs(tok, scorer, fa, fb) > score_pairs(tok, scorer, fb, fa)).mean())
        print(f"test AUC (true next sentence vs wrong pairs): {pair_auc:.3f}")
        print(f"direction accuracy on unseen articles: {direction_acc:.1%}  (50% = guessing)")

        # --- score every pair once, for the evaluation windows and for the 7 real samples ---
        for w in windows:
            w["L"] = pair_logit_matrix(tok, scorer, w["S"])
        for name, sents in sentence_data.items():
            L_samples[name] = pair_logit_matrix(tok, scorer, sents)
        LEARNED_OK = True
    except Exception as e:
        print("Learned ordering model could not be used:", type(e).__name__, str(e)[:300])
        print("-> the rule-based ordering will be used instead")
    finally:
        report_gpu("Stage 2: ordering cross-encoder (training + scoring)")
        tok = scorer = None
        free_gpu()
else:
    print("Skipped (no Wikipedia data).")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

trainable parameters: 21.4 M (word table frozen and kept on the CPU)
training on 32000 sentence pairs for 2 epochs (batch 16) ...
  epoch 1 step 500/2000 loss 1.041 (53s)
  epoch 1 step 1000/2000 loss 1.043 (81s)
  epoch 1 step 1500/2000 loss 1.042 (114s)
  epoch 1 step 2000/2000 loss 1.038 (143s)
epoch 1/2 done | average loss 1.038 | 143s
  epoch 2 step 500/2000 loss 0.970 (171s)
  epoch 2 step 1000/2000 loss 0.970 (215s)
  epoch 2 step 1500/2000 loss 0.968 (253s)
  epoch 2 step 2000/2000 loss 0.968 (282s)
epoch 2/2 done | average loss 0.968 | 282s
test AUC (true next sentence vs wrong pairs): 0.672
direction accuracy on unseen articles: 52.8%  (50% = guessing)
[Stage 2: ordering cross-encoder (training + scoring)]
    PyTorch peak 0.80 GB | whole GPU incl. CUDA context 1.39 GB | limit 1.5 GB -> OK


In [18]:
# ---- compare ordering methods on shuffled windows of unseen articles (real answer key) ----
def kendall_tau(true_positions):
    n = len(true_positions)
    return kendalltau(np.arange(n), true_positions)[0]

def adjacent_accuracy(true_positions):
    tp = np.array(true_positions)
    return float(np.mean(tp[1:] == tp[:-1] + 1))

def evaluate(cfg, wins):
    taus, adjs = [], []
    for w in wins:
        T, s, e = final_scores(w.get("L"), w["E"], w["S"], w["sem"], w["lex"], cfg)
        path, _ = order_group(T, s, e)
        true_pos = w["perm"][path]
        taus.append(kendall_tau(true_pos)); adjs.append(adjacent_accuracy(true_pos))
    return float(np.mean(taus)), float(np.mean(adjs))

if LEARNED_OK:
    configs = {
        "rules only":               dict(use_learned=False),
        "learned only":             dict(use_learned=True, heur_w=0.0, start_w=0.0),
        "learned + start model":    dict(use_learned=True, heur_w=0.0, start_w=0.5),
        "learned + rules":          dict(use_learned=True, heur_w=0.5, start_w=0.0),
        "learned + rules + start":  dict(use_learned=True, heur_w=0.5, start_w=0.5),
    }
    rows = []
    for label, cfg in configs.items():
        tau, adj = evaluate(cfg, tune_w)
        rows.append(dict(method=label, tau=tau))
        print(f"{label:26} tau {tau:6.3f}   adjacent-pair accuracy {adj:.3f}")

    best_label = max(rows, key=lambda r: r["tau"])["method"]
    BEST = configs[best_label]
    print("\nChosen method:", best_label, "->", BEST)

    final_rows = []
    for label in ("rules only", best_label):
        tau, adj = evaluate(configs[label], report_w)
        final_rows.append(dict(method=label, tau=round(tau, 3), adjacent_pair_accuracy=round(adj, 3)))
    rnd = [kendall_tau(np.random.default_rng(i).permutation(WINDOW)) for i in range(200)]
    final_rows.append(dict(method="random order", tau=round(float(np.mean(rnd)), 3),
                           adjacent_pair_accuracy=round(1 / WINDOW, 3)))
    final_table = pd.DataFrame(final_rows).drop_duplicates(subset="method")
    print("\nFINAL RESULT on windows that were NOT used for choosing the method:")
    print(final_table.to_string(index=False))
    final_table.to_csv(os.path.join(OUTPUT_DIR, "ordering_evaluation.csv"), index=False)
    if direction_acc is not None:
        print(f"\nPair model: direction accuracy {direction_acc:.1%}, AUC {pair_auc:.3f} (held-out articles)")
else:
    print("The learned model is not available, the samples are ordered with the rule-based method.")

print("\nOrdering setting for the samples:", BEST)

rules only                 tau  0.007   adjacent-pair accuracy 0.161
learned only               tau -0.032   adjacent-pair accuracy 0.161
learned + start model      tau  0.094   adjacent-pair accuracy 0.153
learned + rules            tau  0.018   adjacent-pair accuracy 0.197
learned + rules + start    tau  0.143   adjacent-pair accuracy 0.161

Chosen method: learned + rules + start -> {'use_learned': True, 'heur_w': 0.5, 'start_w': 0.5}

FINAL RESULT on windows that were NOT used for choosing the method:
                 method   tau  adjacent_pair_accuracy
             rules only 0.071                   0.197
learned + rules + start 0.087                   0.211
           random order 0.020                   0.100

Pair model: direction accuracy 52.8%, AUC 0.672 (held-out articles)

Ordering setting for the samples: {'use_learned': True, 'heur_w': 0.5, 'start_w': 0.5}


In [19]:
ordered = {}   # file -> list of (original_sentence_number, sentence_text) in final order

for name, sents in sentence_data.items():
    T, start_s, end_s = final_scores(L_samples.get(name), vectors[name], sents, semantic[name], lexical[name], BEST)
    labels = clusters[name]

    topic_orders = []
    for c in range(labels.max() + 1):
        members = np.where(labels == c)[0]                       # original indices
        local_T = T[np.ix_(members, members)]
        local_order, _ = order_group(local_T, start_s[members], end_s[members])
        topic_orders.append([int(members[k]) for k in local_order])

    # put the topic whose opening sentence looks most like an opener first
    topic_orders.sort(key=lambda order: start_s[order[0]], reverse=True)

    final_order = [i for order in topic_orders for i in order]
    ordered[name] = [(i + 1, sents[i]) for i in final_order]
    print(f"{name:28} ordered {len(final_order)} sentences in {len(topic_orders)} topic group(s)")

Copy of sample-1_punctuated.txt ordered 49 sentences in 2 topic group(s)
Copy of sample-2_punctuated.txt ordered 49 sentences in 1 topic group(s)
Copy of sample-3_punctuated.txt ordered 48 sentences in 1 topic group(s)
Copy of sample-4_punctuated.txt ordered 50 sentences in 2 topic group(s)
Copy of sample-5_punctuated.txt ordered 51 sentences in 2 topic group(s)
Copy of sample-6_punctuated.txt ordered 35 sentences in 1 topic group(s)
Copy of sample-7_punctuated.txt ordered 48 sentences in 1 topic group(s)


In [20]:
# look at the ordering of one file (number = original position in the jumbled text)
name = list(ordered)[0]
print(name)
for pos, (orig, s) in enumerate(ordered[name][:12], 1):
    print(f"{pos:2d}. (was #{orig:2d}) {s}")

Copy of sample-1_punctuated.txt
 1. (was #27) अयोध्या पवित्र नदी सरयू के तट पर स्थित है।
 2. (was # 3) द्वारका चार धामों में एक धाम भी है।
 3. (was #23) इस रास्ते से भी बनारस सिटी होते हुए छावनी जा सकते हैं।
 4. (was #46) इसका नाम बनारस या वाराणसी भी है।
 5. (was # 7) स्टेशन का नाम कांजीवरम जरूर है किंतु नगर का नाम कांचीपुरम है।
 6. (was #21) ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है।
 7. (was #33) गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा।
 8. (was #20) उस समय गढ़ मुक्तेश्वर बल्लभ संप्रदाय का मुख्य केन्द्र था।
 9. (was # 1) भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।
10. (was # 6) अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है।
11. (was # 8) यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है।
12. (was #37) गंगा किनारे यह भगवान शंकर की प्रसिद्ध पुरी है।


### Quick coherence check

There is no answer key for the real samples, so here is a simple number: the **average similarity between neighbouring sentences**. A good order should score clearly higher than a random order.
(This is the same kind of similarity the search optimises, so it is a sanity check and **not** an accuracy score. The accuracy number is the held-out result above.)

In [21]:
rng = np.random.default_rng(0)

def adjacent_score(order, base):
    order = np.array(order)
    return base[order[:-1], order[1:]].mean()

print(f"{'file':28} {'jumbled input':>14} {'random orders':>14} {'our order':>10}")
for name, sents in sentence_data.items():
    base = 0.6 * np.clip(semantic[name], 0, 1) + 0.4 * lexical[name]
    n = len(sents)
    ours = [orig - 1 for orig, _ in ordered[name]]
    randoms = [adjacent_score(rng.permutation(n), base) for _ in range(50)]
    print(f"{name:28} {adjacent_score(list(range(n)), base):14.3f} {np.mean(randoms):14.3f} {adjacent_score(ours, base):10.3f}")

file                          jumbled input  random orders  our order
Copy of sample-1_punctuated.txt          0.209          0.209      0.429
Copy of sample-2_punctuated.txt          0.225          0.213      0.392
Copy of sample-3_punctuated.txt          0.197          0.201      0.364
Copy of sample-4_punctuated.txt          0.218          0.200      0.351
Copy of sample-5_punctuated.txt          0.226          0.203      0.370
Copy of sample-6_punctuated.txt          0.186          0.171      0.294
Copy of sample-7_punctuated.txt          0.139          0.139      0.273


## 7. Chunks of 5 sentences

The final order (topic groups one after the other) is cut into groups of exactly 5 sentences. Only the very last chunk of a sample can have fewer than 5.

In [22]:
def make_chunks(name):
    items = ordered[name]
    return [items[i:i + CHUNK_SIZE] for i in range(0, len(items), CHUNK_SIZE)]

chunk_data = {name: make_chunks(name) for name in ordered}

for name, chunks in chunk_data.items():
    sizes = [len(c) for c in chunks]
    print(f"{name:28} {len(chunks)} chunks, last chunk has {sizes[-1]} sentence(s)")

Copy of sample-1_punctuated.txt 10 chunks, last chunk has 4 sentence(s)
Copy of sample-2_punctuated.txt 10 chunks, last chunk has 4 sentence(s)
Copy of sample-3_punctuated.txt 10 chunks, last chunk has 3 sentence(s)
Copy of sample-4_punctuated.txt 10 chunks, last chunk has 5 sentence(s)
Copy of sample-5_punctuated.txt 11 chunks, last chunk has 1 sentence(s)
Copy of sample-6_punctuated.txt 7 chunks, last chunk has 5 sentence(s)
Copy of sample-7_punctuated.txt 10 chunks, last chunk has 3 sentence(s)


In [23]:
# save the ordered Hindi text and chunks (before translation)
for name, chunks in chunk_data.items():
    base = name.replace("_punctuated.txt", "")
    with open(os.path.join(OUTPUT_DIR, f"{base}_ordered.txt"), "w", encoding="utf-8") as f:
        for k, chunk in enumerate(chunks, 1):
            f.write(f"[Chunk {k}]\n")
            f.write(" ".join(s for _, s in chunk) + "\n\n")
print("Ordered Hindi chunks saved in", OUTPUT_DIR)

Ordered Hindi chunks saved in /content/drive/MyDrive/Research_Project/Model_2/Sentence_Ordering


## 8. Translation to English (stage 3)

Translators tried so far: `opus-mt-hi-en` was far too weak ("Milky Way" for *Ganga*), and `IndicTrans2` does not load with the newer `transformers` in Colab.
So we use **NLLB-200-distilled-600M**. In fp16 it needs about 1.2 GB for the weights alone, so together with the CUDA context it would not stay under 1.5 GB. It is therefore loaded **quantised (8-bit, then 4-bit)**. The notebook tries the options in this order and keeps the first one that passes a smoke test, a stress test with the longest sentences, **and a real memory measurement under 1.5 GB**:

1. NLLB-600M, 8-bit  2. NLLB-600M, 4-bit  3. M2M100-418M, fp16

**Sentence by sentence, then join 5 per chunk:** small models skip content when given 5 sentences at once.

**Small glossary:** the model translates some names literally (गढ़ = "fort") and some silk words wrongly (कोकून = "coconut"). Before translating, a short Hindi-to-English word list replaces these. This is manual pre-processing based on the sample vocabulary, so mention it in the write-up. `USE_GLOSSARY = False` switches it off for comparison.

In [24]:
import transformers
transformers.logging.set_verbosity_error()      # hide long repeated warnings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB = "facebook/nllb-200-distilled-600M"
M2M = "facebook/m2m100_418M"

def load_nllb(mode):
    tok = AutoTokenizer.from_pretrained(NLLB, src_lang="hin_Deva")
    if mode == "int8":
        from transformers import BitsAndBytesConfig
        model = AutoModelForSeq2SeqLM.from_pretrained(
            NLLB, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map={"": 0})
    elif mode == "4bit":
        from transformers import BitsAndBytesConfig
        model = AutoModelForSeq2SeqLM.from_pretrained(
            NLLB, device_map={"": 0},
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_compute_dtype=torch.float16))
    else:                                        # plain weights (CPU test only)
        model = AutoModelForSeq2SeqLM.from_pretrained(NLLB).to(device)
    model.eval()
    english_id = tok.convert_tokens_to_ids("eng_Latn")

    def translate(sentences):
        inputs = tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=200).to(device)
        with torch.inference_mode():
            out = model.generate(**inputs, forced_bos_token_id=english_id, num_beams=4, max_new_tokens=200)
        return tok.batch_decode(out, skip_special_tokens=True)
    return translate

def load_m2m100():
    tok = AutoTokenizer.from_pretrained(M2M)
    tok.src_lang = "hi"
    model = AutoModelForSeq2SeqLM.from_pretrained(M2M, torch_dtype=torch.float16 if device == "cuda" else torch.float32)
    model = model.to(device).eval()
    english_id = tok.get_lang_id("en")

    def translate(sentences):
        inputs = tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=200).to(device)
        with torch.inference_mode():
            out = model.generate(**inputs, forced_bos_token_id=english_id, num_beams=4, max_new_tokens=200)
        return tok.batch_decode(out, skip_special_tokens=True)
    return translate

if device == "cuda":
    TRIALS = [("NLLB-600M (8-bit)", lambda: load_nllb("int8")),
              ("NLLB-600M (4-bit)", lambda: load_nllb("4bit")),
              ("M2M100-418M (fp16)", load_m2m100)]
else:
    TRIALS = [("NLLB-600M (CPU test)", lambda: load_nllb("plain"))]

SMOKE_TEST = "गंगा नदी हरिद्वार से होकर बहती है।"
all_sents = [s for ss in sentence_data.values() for s in ss]
stress = sorted(all_sents, key=len, reverse=True)[:5]       # the longest sentences = worst case for memory

translate_fn, MT_LABEL = None, None
backup = []                       # models that fit PyTorch's own count but not the whole-GPU count
for label, loader in TRIALS:
    free_gpu()
    candidate = None
    try:
        candidate = loader()
        test_output = candidate([SMOKE_TEST])[0]
        assert len(test_output.split()) >= 3, "empty or too short output"
        candidate(stress)
        row = report_gpu(f"translator trial: {label}")
        if row is not None and not row["whole_gpu_under_limit"]:
            backup.append((row["whole_gpu_gb"], label, loader))
            raise RuntimeError(f"whole GPU {row['whole_gpu_gb']} GB is not under {GPU_LIMIT_GB} GB")
        translate_fn, MT_LABEL = candidate, label
        print(f"Using {label}. Smoke test: {SMOKE_TEST} -> {test_output}")
        break
    except Exception as e:
        print(f"{label} not used: {type(e).__name__}: {str(e)[:160]}")
        candidate = None

if translate_fn is None and backup:
    backup.sort(key=lambda x: x[0])
    whole, label, loader = backup[0]
    print(f"WARNING: no translator is under {GPU_LIMIT_GB} GB when the CUDA context is counted. "
          f"Using {label} ({whole} GB); PyTorch's own count is under the limit.")
    free_gpu()
    translate_fn, MT_LABEL = loader(), label

assert translate_fn is not None, "No translation model could be loaded"
free_gpu()      # reset the peak counter, the full translation run is measured below

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (18 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.94GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

[translator trial: M2M100-418M (fp16)]
    PyTorch peak 1.14 GB | whole GPU incl. CUDA context 1.38 GB | limit 1.5 GB -> OK
Using M2M100-418M (fp16). Smoke test: गंगा नदी हरिद्वार से होकर बहती है। -> The river is flowing from the river.


In [25]:
# Hindi -> English word list for things the model gets wrong (longest entries are applied first)
GLOSSARY = {
    # places and names
    "गढ़ मुक्तेश्वर": "Garhmukteshwar", "गढ़मुक्तेश्वर": "Garhmukteshwar",
    "मुक्तेश्वर महादेव": "Mukteshwar Mahadev", "शिव बल्लभपुर": "Shiv Ballabhpur",
    "लक्सर": "Laksar", "कांजीवरम": "Kanjivaram", "कांचीपुरम": "Kanchipuram",
    "हस्तिनापुर": "Hastinapur", "तिजारा": "Tijara", "मीराबाई": "Mirabai",
    "गंगा मैया": "Ganga Maiya", "अक्षय तृतीया": "Akshaya Tritiya",
    "कार्तिक पूर्णिमा": "Kartik Purnima", "परशुराम": "Parshuram",
    "श्रेयंषकुमार": "Shreyansh Kumar", "आदिनाथ": "Adinath",
    "महावीर स्वामी": "Mahavir Swami", "पार्श्वनाथ": "Parshvanath",
    "मुगलसराय": "Mughalsarai", "बल्लभ संप्रदाय": "Vallabh sect",
    # silk and cotton words
    "रेशम के कीड़ों": "silkworms", "रेशम के कीड़े": "silkworms", "रेशम का कीड़ा": "silkworm",
    "रेशम कीट": "silkworm", "कोकून": "cocoon", "कोये": "cocoons", "कोया": "cocoon",
    "टसर": "tussar", "मूंगा": "muga", "ऐरी": "eri", "शहतूत": "mulberry",
    "क्राइसेलिस": "chrysalis", "बीजकोष": "cotton boll",
}
# same Unicode spelling as the cleaned text (see section 2), otherwise some words would not match
GLOSSARY = {unicodedata.normalize("NFC", k): v for k, v in GLOSSARY.items()}
GLOSSARY_KEYS = sorted(GLOSSARY, key=len, reverse=True)

def apply_glossary(text):
    if not USE_GLOSSARY:
        return text
    for key in GLOSSARY_KEYS:
        text = text.replace(key, GLOSSARY[key])
    return text

def tidy_english(text):
    text = re.sub(r"\s+'s\b", "'s", text)            # "Shivaji 's" -> "Shivaji's"
    text = re.sub(r"\s+([,.;:?!])", r"\1", text)     # no space before punctuation
    return text.strip()

def translate_texts(texts, batch_size=5):
    results = []
    for start in range(0, len(texts), batch_size):
        results.extend(translate_fn(texts[start:start + batch_size]))
    return results

def translate_chunk(chunk):
    sentences = [apply_glossary(s) for _, s in chunk]

    if TRANSLATE_MODE == "chunk":
        hindi_text = " ".join(sentences)
        english = translate_texts([hindi_text])[0]
        ratio = len(english.split()) / max(1, len(hindi_text.split()))
        if ratio >= 0.8:                      # looks complete
            return tidy_english(english), "whole chunk"
        # too short -> the model probably dropped sentences, do it one by one

    english = " ".join(t.strip() for t in translate_texts(sentences))
    return tidy_english(english), "sentence-by-sentence"

# quick test on the first chunk
first_name = list(chunk_data)[0]
test_en, how = translate_chunk(chunk_data[first_name][0])
print("Hindi:  ", " ".join(s for _, s in chunk_data[first_name][0]))
print("English:", test_en)
print("Method :", how)

Hindi:   अयोध्या पवित्र नदी सरयू के तट पर स्थित है। द्वारका चार धामों में एक धाम भी है। इस रास्ते से भी बनारस सिटी होते हुए छावनी जा सकते हैं। इसका नाम बनारस या वाराणसी भी है। स्टेशन का नाम कांजीवरम जरूर है किंतु नगर का नाम कांचीपुरम है।
English: Ayodhya is located on the shore of the sacred river Siru. There is also a fire in the four corners. This way you can also go to Banaras City. Its name is Banaras or Varanasi. The station’s name is Kanjivaram but the city’s name is Kanchipuram.
Method : sentence-by-sentence


In [26]:
results = {}   # file -> list of dicts (one per chunk)

for name, chunks in chunk_data.items():
    results[name] = []
    for k, chunk in enumerate(chunks, 1):
        english, how = translate_chunk(chunk)
        results[name].append({
            "chunk": k,
            "sentence_numbers": [orig for orig, _ in chunk],
            "hindi": " ".join(s for _, s in chunk),
            "english": english,
            "method": how,
            "model": MT_LABEL,
            "glossary_used": USE_GLOSSARY,
        })
    print(f"{name:28} {len(chunks)} chunks translated")

report_gpu("Translation (" + MT_LABEL + ")")

Copy of sample-1_punctuated.txt 10 chunks translated
Copy of sample-2_punctuated.txt 10 chunks translated
Copy of sample-3_punctuated.txt 10 chunks translated
Copy of sample-4_punctuated.txt 10 chunks translated
Copy of sample-5_punctuated.txt 11 chunks translated
Copy of sample-6_punctuated.txt 7 chunks translated
Copy of sample-7_punctuated.txt 10 chunks translated
[Translation (M2M100-418M (fp16))]
    PyTorch peak 1.13 GB | whole GPU incl. CUDA context 1.40 GB | limit 1.5 GB -> OK


{'stage': 'Translation (M2M100-418M (fp16))',
 'pytorch_peak_gb': 1.127,
 'whole_gpu_gb': 1.397,
 'pytorch_under_limit': True,
 'whole_gpu_under_limit': True}

## 9. Save the final outputs

In [27]:
for name, rows in results.items():
    base = name.replace("_punctuated.txt", "")

    # readable text file: Hindi + English for every chunk
    with open(os.path.join(OUTPUT_DIR, f"{base}_final.txt"), "w", encoding="utf-8") as f:
        for r in rows:
            f.write(f"[Chunk {r['chunk']}]  (original sentence numbers: {r['sentence_numbers']})\n")
            f.write("HI: " + r["hindi"] + "\n")
            f.write("EN: " + r["english"] + "\n\n")

    # json for later use
    with open(os.path.join(OUTPUT_DIR, f"{base}_final.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

print("Saved", len(results), "files x 2 formats in", OUTPUT_DIR)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print("  ", fname)

# ---- cells a human should double-check (long merged sentences, tiny fragments) ----
flags = []
for name, sents in sentence_data.items():
    for k, s in enumerate(sents, 1):
        why = []
        if len(s.split()) > LONG_SENTENCE:
            why.append("possible_merged_sentence")
        if len(s.split()) < 4:
            why.append("short_fragment")
        if not re.search(r"[।?!.]$", s):
            why.append("no_terminal_punctuation")
        if why:
            flags.append(dict(file=name, original_id=k, reasons=";".join(why), sentence=s))
pd.DataFrame(flags, columns=["file", "original_id", "reasons", "sentence"]).to_csv(
    os.path.join(OUTPUT_DIR, "review_flags.csv"), index=False, encoding="utf-8-sig")
print("review_flags.csv:", len(flags), "sentences to double-check")

run_info = dict(ordering_setting=BEST, learned_ordering_used=bool(LEARNED_OK), translation_model=MT_LABEL,
                translate_mode=TRANSLATE_MODE, glossary_used=USE_GLOSSARY, chunk_size=CHUNK_SIZE, device=device,
                gpu_limit_gb=GPU_LIMIT_GB)
with open(os.path.join(OUTPUT_DIR, "run_info.json"), "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2, default=str)

Saved 7 files x 2 formats in /content/drive/MyDrive/Research_Project/Model_2/Sentence_Ordering
   Copy of sample-1_final.json
   Copy of sample-1_final.txt
   Copy of sample-1_ordered.txt
   Copy of sample-2_final.json
   Copy of sample-2_final.txt
   Copy of sample-2_ordered.txt
   Copy of sample-3_final.json
   Copy of sample-3_final.txt
   Copy of sample-3_ordered.txt
   Copy of sample-4_final.json
   Copy of sample-4_final.txt
   Copy of sample-4_ordered.txt
   Copy of sample-5_final.json
   Copy of sample-5_final.txt
   Copy of sample-5_ordered.txt
   Copy of sample-6_final.json
   Copy of sample-6_final.txt
   Copy of sample-6_ordered.txt
   Copy of sample-7_final.json
   Copy of sample-7_final.txt
   Copy of sample-7_ordered.txt
   ordering_evaluation.csv
review_flags.csv: 10 sentences to double-check


In [28]:
# preview: first two chunks of every sample
for name, rows in results.items():
    print("\n" + "=" * 70, "\n", name)
    for r in rows[:2]:
        print(f"\n[Chunk {r['chunk']}]")
        print("HI:", r["hindi"][:200], "...")
        print("EN:", r["english"][:300])


 Copy of sample-1_punctuated.txt

[Chunk 1]
HI: अयोध्या पवित्र नदी सरयू के तट पर स्थित है। द्वारका चार धामों में एक धाम भी है। इस रास्ते से भी बनारस सिटी होते हुए छावनी जा सकते हैं। इसका नाम बनारस या वाराणसी भी है। स्टेशन का नाम कांजीवरम जरूर है कि ...
EN: Ayodhya is located on the shore of the sacred river Siru. There is also a fire in the four corners. This way you can also go to Banaras City. Its name is Banaras or Varanasi. The station’s name is Kanjivaram but the city’s name is Kanchipuram.

[Chunk 2]
HI: ज्ञात रहे कि शिव बल्लभपुर को ही अब गढ़ मुक्तेश्वर के नाम से जाना जाता है। गढ़ मुक्तेश्वर का नाम शिव बल्लभपुर पड़ा। उस समय गढ़ मुक्तेश्वर बल्लभ संप्रदाय का मुख्य केन्द्र था। भगवान राम के पूर्वज महाराज  ...
EN: Shiv Ballabhpur is now known as Garhmukteshwar. Garhmukteshwar was named Shiv Ballabhpur. At that time, Garhmukteshwar was the main center of the Vallabh sect. God Ram's ancestor Maharaj Shivi had spent his vanprst in Garhmukteshwar. Ayodhya is the holy land of the avatar o

## 10. GPU memory report (the 1.5 GB rule)

In [29]:
gpu_df = pd.DataFrame(gpu_log)
if len(gpu_df):
    gpu_df.to_csv(os.path.join(OUTPUT_DIR, "gpu_memory_report.csv"), index=False)
    print(gpu_df.to_string(index=False))
    final_stages = gpu_df[~gpu_df["stage"].str.contains("trial")]
    all_torch = bool(final_stages["pytorch_under_limit"].all())
    all_whole = bool(final_stages["whole_gpu_under_limit"].all())
    print(f"\nEvery model, PyTorch's own count under {GPU_LIMIT_GB} GB: {all_torch}")
    print(f"Every model, whole GPU including the CUDA context under {GPU_LIMIT_GB} GB: {all_whole}")
else:
    print("No GPU measurements (CPU run).")

                                               stage  pytorch_peak_gb  whole_gpu_gb  pytorch_under_limit  whole_gpu_under_limit
                   Stage 1: MiniLM bi-encoder (fp16)            0.618         0.761                 True                   True
Stage 2: ordering cross-encoder (training + scoring)            0.798         1.393                 True                   True
                translator trial: M2M100-418M (fp16)            1.143         1.376                 True                   True
                    Translation (M2M100-418M (fp16))            1.127         1.397                 True                   True

Every model, PyTorch's own count under 1.5 GB: True
Every model, whole GPU including the CUDA context under 1.5 GB: True


## 11. Notes for the write-up

* **Constraints:** no off-the-shelf LLM. Three small models, loaded one at a time: MiniLM (fp16), our own MiniLM cross-encoder for ordering, NLLB-600M quantised for translation. A hard 1.5 GB limit is set on PyTorch, and `gpu_memory_report.csv` has the measured peak per stage (PyTorch count and whole GPU including the CUDA context).
* **Ordering:** MiniLM + TF-IDF similarity, Hindi discourse rules, and a **learned pairwise model** trained on correctly ordered Hindi Wikipedia text. The method was chosen on one half of held-out shuffled windows and reported on the other half (`ordering_evaluation.csv`). Quote those numbers, not the neighbour-similarity score (that one is circular).
* **Cleaning:** `NULL` removed; broken numbers fixed; one Unicode spelling per letter (the files mixed two encodings of ढ़).
* **Translation:** NLLB-600M, quantised, sentence by sentence (small models drop content on long inputs). A short manual glossary fixes names and silk words; say so, and compare with `USE_GLOSSARY = False` if you want to show its effect.
* **Limitations:** the learned model is trained on encyclopedia text, so it is weaker on dialogue (samples 6 and 7); the real samples have no answer key, so their order is judged by reading plus the held-out evaluation; sentences that Task 1 merged hurt ordering and translation (see `review_flags.csv`); quantisation can slightly lower translation quality; Hindi proper nouns can still be mistranslated.
* **Task 1 (punctuation) model:** Cadence-Fast has about 0.3 B parameters and its weights are stored in FP32 (roughly 1.1 to 1.2 GB). Loaded as float32 it is close to the 1.5 GB limit once the CUDA context is added. Load it in half precision and print the memory in the Task 1 notebook. The `cadence-punctuation` package itself defaults to bfloat16; if that does not run on your GPU use float16, and in both cases check that the punctuation output is unchanged:

```python
model = AutoModel.from_pretrained(TOKENIZER_DIR, trust_remote_code=True, torch_dtype=torch.bfloat16, local_files_only=True).to("cuda").eval()
print("PyTorch allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
free, total = torch.cuda.mem_get_info(); print("Whole GPU in use:", (total - free) / 1024**3, "GB")
```